# MTurk Task B Mining

This notebook mines **all** matching Task B commitment examples and writes them to CSV/JSON.
The later `selecter.ipynb` notebook filters these down to 100 rows after checking whether `gpt-4o-mini`
can answer both the action and commitment-sentence questions correctly.


In [ ]:

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path('/playpen-ssd/smerrill/deception2')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.append(str(SRC_ROOT))

from mturk_dataset_utils import (
    DATASETS_ROOT,
    ENVIRONMENTS,
    MODEL_VARIANTS,
    MTURK_CACHE_ROOT,
    MTURK_OUTPUT_ROOT,
    build_taskb_dataframe,
    save_task_dataframe,
    taskb_summary_dataframe,
)

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)


In [ ]:

# Configuration
MAX_EXAMPLES_PER_ENV_TO_LOAD = None  # Use None to mine every localization file
LARGE_SPIKE_DELTA_THRESHOLD = 0.50
MIN_FULL_SENTENCES = 5
MAX_FULL_SENTENCES = 12
ONE_TASK_PER_EXAMPLE = False  # Save all matching examples, not just one per example
CACHE_ROOT = MTURK_CACHE_ROOT / 'taskb_mining'
REFRESH_MINING_CACHE = False
PROGRESS_EVERY_FILES = 250
WRITE_JSON_EXPORT = True

CSV_PATH = MTURK_OUTPUT_ROOT / 'taskb.csv'
JSON_PATH = MTURK_OUTPUT_ROOT / 'taskb.json'
SUMMARY_PATH = MTURK_OUTPUT_ROOT / 'taskb_summary.csv'

print('Configuration')
print(f'  Dataset root: {DATASETS_ROOT}')
print(f'  Output root: {MTURK_OUTPUT_ROOT}')
print(f'  Cache root: {CACHE_ROOT}')
print(f'  Models: {list(MODEL_VARIANTS.keys())}')
print(f'  Environments: {ENVIRONMENTS}')
print(f'  Max examples per env: {MAX_EXAMPLES_PER_ENV_TO_LOAD}')
print(f'  Spike threshold: {LARGE_SPIKE_DELTA_THRESHOLD}')
print(f'  Sentence filter: [{MIN_FULL_SENTENCES}, {MAX_FULL_SENTENCES}]')
print(f'  One task per example: {ONE_TASK_PER_EXAMPLE}')
print(f'  Refresh mining cache: {REFRESH_MINING_CACHE}')
print(f'  Progress every files: {PROGRESS_EVERY_FILES}')
print(f'  Write JSON export: {WRITE_JSON_EXPORT}')


In [ ]:

taskb_df = build_taskb_dataframe(
    dataset_root=DATASETS_ROOT,
    model_variants=MODEL_VARIANTS,
    environments=ENVIRONMENTS,
    threshold=LARGE_SPIKE_DELTA_THRESHOLD,
    min_full_sentences=MIN_FULL_SENTENCES,
    max_full_sentences=MAX_FULL_SENTENCES,
    max_examples_per_env_to_load=MAX_EXAMPLES_PER_ENV_TO_LOAD,
    one_task_per_example=ONE_TASK_PER_EXAMPLE,
    cache_root=CACHE_ROOT,
    refresh_cache=REFRESH_MINING_CACHE,
    progress_every_files=PROGRESS_EVERY_FILES,
)
taskb_summary_df = taskb_summary_dataframe(taskb_df)

print(f'Mined {len(taskb_df)} Task B rows')
display(taskb_summary_df)
display(taskb_df.head(10))


In [ ]:
if not taskb_df.empty:
    preview_cols = [
        'task_id', 'model_id', 'environment', 'example_id', 'sentence_idx',
        'full_reasoning_num_sentences', 'gold_action_value',
        'gold_commitment_option_value', 'gold_action_share', 'spike_delta',
    ]
    display(taskb_df[preview_cols].head(20))
else:
    print('No Task B rows found with the current settings.')


In [ ]:

save_task_dataframe(
    taskb_df,
    csv_path=CSV_PATH,
    json_path=JSON_PATH if WRITE_JSON_EXPORT else None,
    summary_df=taskb_summary_df,
    summary_path=SUMMARY_PATH,
)

print(f'Saved CSV to:     {CSV_PATH}')
if WRITE_JSON_EXPORT:
    print(f'Saved JSON to:    {JSON_PATH}')
else:
    print('Skipped JSON export')
print(f'Saved summary to: {SUMMARY_PATH}')
